<a href="https://colab.research.google.com/github/shay2222/ml-systems-lab/blob/main/05-distributed/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nproc

2


In [2]:
%%writefile ddp_test.py
import time
import torch
import torch.nn as nn
import torch.distributed as dist

def main():
    dist.init_process_group("gloo")
    rank = dist.get_rank()
    world = dist.get_world_size()

    torch.manual_seed(0)
    model = nn.Sequential(
        nn.Linear(512, 1024), nn.ReLU(),
        nn.Linear(1024, 1024), nn.ReLU(),
        nn.Linear(1024, 10),
    )
    ddp = nn.parallel.DistributedDataParallel(model)
    opt = torch.optim.SGD(ddp.parameters(), lr=0.01)
    loss_fn = nn.CrossEntropyLoss()

    batch = 64  # per process, so the total batch grows with world size
    x = torch.randn(batch, 512)
    y = torch.randint(0, 10, (batch,))

    def step():
        opt.zero_grad()
        loss = loss_fn(ddp(x), y)
        loss.backward()   # gradients are averaged across processes here
        opt.step()

    for _ in range(5):    # warm-up
        step()

    dist.barrier()
    start = time.time()
    for _ in range(50):
        step()
    dist.barrier()
    step_time = (time.time() - start) / 50

    # Communication cost on its own: all-reduce a buffer the size of the gradients
    n_params = sum(p.numel() for p in model.parameters())
    buf = torch.randn(n_params)
    dist.barrier()
    start = time.time()
    for _ in range(50):
        dist.all_reduce(buf)
    dist.barrier()
    comm_time = (time.time() - start) / 50

    if rank == 0:
        samples_per_sec = world * batch / step_time
        print(f"world_size={world} params={n_params} "
              f"step_ms={step_time*1000:.2f} allreduce_ms={comm_time*1000:.2f} "
              f"samples_per_sec={samples_per_sec:.0f}")

    dist.destroy_process_group()

if __name__ == "__main__":
    main()

Writing ddp_test.py


Run with one process (baseline)

In [3]:
!torchrun --standalone --nproc_per_node=1 ddp_test.py 2>&1 | grep world_size

world_size=1 params=1585162 step_ms=23.99 allreduce_ms=0.07 samples_per_sec=2668


Rune with two process

In [4]:
!torchrun --standalone --nproc_per_node=2 ddp_test.py 2>&1 | grep world_size

world_size=2 params=1585162 step_ms=37.82 allreduce_ms=6.86 samples_per_sec=3384
